# BreaQ 2026 — Challenge 1: Trust No Input

**Layer:** application and circuit construction  
**Difficulty:** beginner core, intermediate exploit, advanced extension  
**Recommended time:** 3–5 hours  
**Team roles:** investigator, exploit developer, defense analyst

> A BreaQ Cloud customer reports that a quantum policy-fingerprint service sometimes rejects valid assets after migration. Operations claims that the quantum circuit is unreliable. Your task is to determine whether the failure is quantum noise, incorrect circuit construction, or unsafe application-layer interpretation.

This challenge runs entirely on a local ideal simulator. No cloud account or real device is required.

## Rules and deliverables

All supplied local artifacts are in scope. Reading package source is allowed, but **finding a suspicious line is not sufficient evidence**. Your team must demonstrate the failure experimentally and provide a generalized mitigation.

Submit:

1. A threat model and one-sentence security claim.
2. A minimal reproducer.
3. Evidence localizing the fault.
4. A secure decoder or validation wrapper.
5. At least one negative test showing your defense does not merely hard-code the demonstration cases.
6. `submission.json`, generated near the end of this notebook.

External assistants and documentation may be used, but every member must be able to defend the submitted reasoning.

## Architecture under investigation

```text
Asset registry ── expected fingerprint
                         │
Manifest ──► circuit builder ──► ideal backend ──► raw counts
   │                                              │
   └──────── compatibility metadata ──► report adapter ──► MATCH / MISMATCH
```

### Contract

- Four logical slots are ordered as `alpha, beta, gamma, delta`.
- Slot `i` is encoded on Qiskit qubit `i`.
- The registry fingerprint is written in the canonical logical-slot order.
- Legacy compatibility metadata may affect presentation, but it must not silently change a security decision.

Your first job is to decide which data crosses a trust boundary and which representation is authoritative.

In [1]:
from pathlib import Path
import json
from itertools import permutations

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

from breaq_ch1 import ChallengeService

INSTANCE = Path("team_instance.json")
service = ChallengeService(INSTANCE)
print("Team:", service.team_id)
print("Cases:", [case.case_id for case in service.list_cases()])

Team: DEMO-TEAM
Cases: ['CAL-ALPHA', 'CAL-BETA', 'MIG-DELTA', 'INCIDENT-7']


## Warm-up: Qiskit bit order

Qiskit labels qubits from `q0` upward, while displayed classical count strings place the highest-index classical bit on the left. That distinction is harmless when explicitly converted and dangerous when separate components silently assume different conventions.

Run the two-qubit example and explain why the visual circuit and count key are consistent.

In [28]:
warmup = QuantumCircuit(2, 2)
warmup.x(0)
warmup.measure([0, 1], [0, 1])
print(warmup.draw(output="text"))
print("Expected displayed key: 01")

     ┌───┐┌─┐┌─┐
q_0: ┤ X ├┤M├┤M├
     └───┘└╥┘└╥┘
q_1: ──────╫──╫─
           ║  ║ 
c: 2/══════╩══╩═
           0  1 
Expected displayed key: 01


**Warm-up question:** If the displayed count key is `0110`, what is the corresponding sequence in the canonical order `q0, q1, q2, q3`?

In [29]:
# TODO: replace None with your answer and explanation.
warmup_canonical = "0110"
warmup_explanation = ("Qiskit prints the highest classical bit on the left (c3 c2 c1 c0), "
                      "so the canonical q0..q3 order is the key read right-to-left: "
                      ".It is a palindrome, so reversing changes nothing.")

## Phase A — Triage

Execute every case. Start with observations, not assumptions. Record the expected registry value, reported value, decision, dominant raw key, and provenance.

In [41]:
case_ids = [case.case_id for case in service.list_cases()]
results = {case_id: service.execute(case_id) for case_id in case_ids}

for case_id, result in results.items():
    print(f"{case_id:12} expected={result['registry_fingerprint']} "
          f"reported={result['reported_fingerprint']} "
          f"decision={result['decision']:8} counts={result['counts']}")

CAL-ALPHA    expected=1001 reported=1001 decision=MATCH    counts={'1001': 512}
CAL-BETA     expected=0110 reported=0110 decision=MATCH    counts={'0110': 512}
MIG-DELTA    expected=1010 reported=0110 decision=MISMATCH counts={'0101': 512}
INCIDENT-7   expected=1100 reported=0101 decision=MISMATCH counts={'0011': 512}


In [43]:
# TODO: turn the observations into a compact evidence table or dictionary.
evidence_table = []
for case in service.list_cases():
    r = results[case.case_id]
    key = max(r["counts"], key=r["counts"].get)
    evidence_table.append({
        "case_id": case.case_id,
        "expected": r["registry_fingerprint"],
        "reported": r["reported_fingerprint"],
        "decision": r["decision"],
        "dominant_raw_key": key,
        "raw_key_share": r["counts"][key] / sum(r["counts"].values()),   # 1.0 = deterministic
        "provenance": case.provenance,
    })
evidence_table

[{'case_id': 'CAL-ALPHA',
  'expected': '1001',
  'reported': '1001',
  'decision': 'MATCH',
  'dominant_raw_key': '1001',
  'raw_key_share': 1.0,
  'provenance': 'factory-calibration'},
 {'case_id': 'CAL-BETA',
  'expected': '0110',
  'reported': '0110',
  'decision': 'MATCH',
  'dominant_raw_key': '0110',
  'raw_key_share': 1.0,
  'provenance': 'factory-calibration'},
 {'case_id': 'MIG-DELTA',
  'expected': '1010',
  'reported': '0110',
  'decision': 'MISMATCH',
  'dominant_raw_key': '0101',
  'raw_key_share': 1.0,
  'provenance': 'migration-adapter'},
 {'case_id': 'INCIDENT-7',
  'expected': '1100',
  'reported': '0101',
  'decision': 'MISMATCH',
  'dominant_raw_key': '0011',
  'raw_key_share': 1.0,
  'provenance': 'external-import'}]

### Threat model

Complete this before inspecting implementation details.

| Element | Your answer |
|---|---|
| Protected asset | |
| Required property | |
| Adversary capability | |
| Trusted input | |
| Untrusted input | |
| Observable symptom | |
| Initial alternative explanations | |

In [45]:
threat_model = {
    "asset": "the MATCH/MISMATCH authorization decision for each asset",
    "security_property": "integrity: the decision depends only on the measured fingerprint and the registry",
    "adversary_capability": "can edit an imported manifest (compatibility_map, provenance), not the registry, circuit or simulator",
    "trusted_inputs": ["registry_fingerprint", "circuit built by the service", "simulator counts"],
    "untrusted_inputs": ["compatibility_map", "provenance", "schema"],
    "alternative_explanations": ["quantum noise", "wrong circuit construction", "Qiskit bit-order mistake", "unsafe report adapter"],
}
threat_model

{'asset': 'the MATCH/MISMATCH authorization decision for each asset',
 'security_property': 'integrity: the decision depends only on the measured fingerprint and the registry',
 'adversary_capability': 'can edit an imported manifest (compatibility_map, provenance), not the registry, circuit or simulator',
 'trusted_inputs': ['registry_fingerprint',
  'circuit built by the service',
  'simulator counts'],
 'untrusted_inputs': ['compatibility_map', 'provenance', 'schema'],
 'alternative_explanations': ['quantum noise',
  'wrong circuit construction',
  'Qiskit bit-order mistake',
  'unsafe report adapter']}

## Phase B — Localize the failure

For each incident, inspect three boundaries independently:

1. The circuit prepared for execution.
2. The raw backend output.
3. The application report.

Useful methods include drawing the circuit, using `Statevector` on a measurement-free copy, examining the manifest, and comparing equivalent representations. Do not assume that `reported_fingerprint` is raw quantum output.

In [46]:
focus_case = "MIG-DELTA"
manifest = service.get_manifest(focus_case)
circuit = service.get_circuit(focus_case)

print(json.dumps(manifest, indent=2))
print(circuit.draw(output="text"))

{
  "case_id": "MIG-DELTA",
  "schema": "legacy-0.9",
  "logical_slots": [
    "alpha",
    "beta",
    "gamma",
    "delta"
  ],
  "compatibility_map": [
    1,
    0,
    2,
    3
  ],
  "provenance": "migration-adapter"
}
     ┌───┐      ░            ░ ┌───┐┌─┐         
q_0: ┤ H ├──────░───■────────░─┤ H ├┤M├─────────
     ├───┤      ░   │        ░ ├───┤└╥┘┌─┐      
q_1: ┤ H ├──────░───┼────────░─┤ H ├─╫─┤M├──────
     ├───┤      ░   │        ░ ├───┤ ║ └╥┘┌─┐   
q_2: ┤ H ├──────░───┼────■───░─┤ H ├─╫──╫─┤M├───
     ├───┤      ░   │    │   ░ ├───┤ ║  ║ └╥┘┌─┐
q_3: ┤ H ├──────░───┼────┼───░─┤ H ├─╫──╫──╫─┤M├
     ├───┤┌───┐ ░ ┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║  ║ └╥┘
q_4: ┤ X ├┤ H ├─░─┤ X ├┤ X ├─░───────╫──╫──╫──╫─
     └───┘└───┘ ░ └───┘└───┘ ░       ║  ║  ║  ║ 
c: 4/════════════════════════════════╩══╩══╩══╩═
                                     0  1  2  3 


In [47]:
# TODO: inspect at least one additional case and write code that localizes the fault.
# Suggested outputs: dominant raw key, canonical interpretation, reported interpretation.
localization_evidence = {}
for case in service.list_cases():
    cid = case.case_id
    qc = service.get_circuit(cid)
    # 1) circuit: which data qubits control a CX -> the encoded fingerprint
    encoded = ["0"] * 4
    for inst in qc.data:
        if inst.operation.name == "cx":
            encoded[qc.find_bit(inst.qubits[0]).index] = "1"
    encoded = "".join(encoded)
    sv = Statevector(qc.remove_final_measurements(inplace=False))
    ideal = {str(k): float(v) for k, v in sv.probabilities_dict(qargs=[0, 1, 2, 3], decimals=6).items()}
    # 2) raw backend output
    r = service.execute(cid)
    raw_key = max(r["counts"], key=r["counts"].get)
    canonical = raw_key[::-1]
    # 3) application report
    reported, registry = r["reported_fingerprint"], r["registry_fingerprint"]
    localization_evidence[cid] = {
        "registry": registry, "circuit_encodes": encoded, "ideal_probabilities": ideal,
        "raw_key": raw_key, "canonical_from_raw": canonical, "reported": reported,
        "map": service.get_manifest(cid)["compatibility_map"],
        "first_wrong_boundary": ("circuit" if encoded != registry else
                                 "raw output" if canonical != registry else
                                 "report" if reported != registry else "none"),
    }
localization_evidence

{'CAL-ALPHA': {'registry': '1001',
  'circuit_encodes': '1001',
  'ideal_probabilities': {'1001': 1.0},
  'raw_key': '1001',
  'canonical_from_raw': '1001',
  'reported': '1001',
  'map': [0, 1, 2, 3],
  'first_wrong_boundary': 'none'},
 'CAL-BETA': {'registry': '0110',
  'circuit_encodes': '0110',
  'ideal_probabilities': {'0110': 1.0},
  'raw_key': '0110',
  'canonical_from_raw': '0110',
  'reported': '0110',
  'map': [0, 1, 2, 3],
  'first_wrong_boundary': 'none'},
 'MIG-DELTA': {'registry': '1010',
  'circuit_encodes': '1010',
  'ideal_probabilities': {'0101': 1.0},
  'raw_key': '0101',
  'canonical_from_raw': '1010',
  'reported': '0110',
  'map': [1, 0, 2, 3],
  'first_wrong_boundary': 'report'},
 'INCIDENT-7': {'registry': '1100',
  'circuit_encodes': '1100',
  'ideal_probabilities': {'0011': 1.0},
  'raw_key': '0011',
  'canonical_from_raw': '1100',
  'reported': '0101',
  'map': [3, 1, 2, 0],
  'first_wrong_boundary': 'report'}}

### Your causal claim

State one falsifiable sentence, for example: *Changing X while holding Y and Z constant changes W.* Then demonstrate it in code.

In [12]:
causal_claim = ""
# TODO: controlled experiment supporting or refuting the claim.
controlled_experiment = {}
controlled_experiment

{}

<details>
<summary>Hint 1 (small)</summary>

Compare `registry_fingerprint`, the dominant count key, and `reported_fingerprint` as three distinct representations.
</details>

<details>
<summary>Hint 2 (medium)</summary>

Hold the encoded fingerprint fixed and vary only the compatibility metadata with `service.simulate_adapter(...)`.
</details>

<details>
<summary>Hint 3 (large)</summary>

Convert a displayed Qiskit count key into `q0, q1, ...` order before applying any business logic. Ask whether a legacy display permutation should participate in authorization at all.
</details>

## Phase C — Minimal reproducer

Produce the smallest example that changes the application result without changing the encoded quantum fingerprint. A good reproducer should have:

- one source fingerprint;
- two metadata configurations;
- identical canonical quantum results;
- different application reports.

In [49]:
def minimal_reproducer(service):
    """Return evidence for the same circuit semantics under two metadata choices."""
    fingerprint = "1010"
    configs = {"identity_map": [0, 1, 2, 3], "legacy_map": [1, 0, 2, 3]}
    runs = {name: service.simulate_adapter(fingerprint, m) for name, m in configs.items()}
    a, b = runs["identity_map"], runs["legacy_map"]
    return {
        "source_fingerprint": fingerprint,
        "metadata_configs": configs,
        "counts": {name: r["counts"] for name, r in runs.items()},
        "canonical": {name: r["canonical_fingerprint"] for name, r in runs.items()},
        "reported": {name: r["reported_fingerprint"] for name, r in runs.items()},
        "decision": {name: "MATCH" if r["reported_fingerprint"] == fingerprint else "MISMATCH"
                     for name, r in runs.items()},
        "same_counts": a["counts"] == b["counts"],
        "same_canonical": a["canonical_fingerprint"] == b["canonical_fingerprint"],
        "different_reports": a["reported_fingerprint"] != b["reported_fingerprint"],
    }

reproducer = minimal_reproducer(service)
reproducer

{'source_fingerprint': '1010',
 'metadata_configs': {'identity_map': [0, 1, 2, 3],
  'legacy_map': [1, 0, 2, 3]},
 'counts': {'identity_map': {'0101': 512}, 'legacy_map': {'0101': 512}},
 'canonical': {'identity_map': '1010', 'legacy_map': '1010'},
 'reported': {'identity_map': '1010', 'legacy_map': '0110'},
 'decision': {'identity_map': 'MATCH', 'legacy_map': 'MISMATCH'},
 'same_counts': True,
 'same_canonical': True,
 'different_reports': True}

## Phase D — Defensive fix

Implement a trusted decoder for security decisions. It must work for fingerprints not present in the demonstration instance.

**Requirement:** Legacy presentation may still exist, but the authorization comparison must use canonical logical-slot order.

In [50]:
from breaq_ch1 import compatibility_decode


def secure_decode(counts: dict[str, int]) -> str:
    """Return the fingerprint in canonical q0..q(n-1) order."""
    # Reads only the raw counts: no case ID, no registry value, no compatibility_map.
    if not counts:
        raise ValueError("no counts")
    keys = {k.replace(" ", ""): v for k, v in counts.items()}
    if any(not k or set(k) - set("01") for k in keys):
        raise ValueError("count keys must be bit strings")
    if len({len(k) for k in keys}) != 1:
        raise ValueError("mixed key lengths")
    top_key, top = max(keys.items(), key=lambda kv: kv[1])
    if 2 * top <= sum(keys.values()):
        raise ValueError("no strict-majority outcome")   # fail closed on ties and heavy noise
    return top_key[::-1]                                  # Qiskit display order -> q0..q(n-1)


def secure_decision(counts: dict[str, int], registry_fingerprint: str) -> str:
    """Authorization uses canonical order only; any doubt -> MISMATCH (fail closed)."""
    try:
        return "MATCH" if secure_decode(counts) == registry_fingerprint else "MISMATCH"
    except ValueError:
        return "MISMATCH"


def secure_report(counts, registry_fingerprint, compatibility_map):
    """Decision first, from canonical order; the legacy map only formats the display."""
    decision = secure_decision(counts, registry_fingerprint)
    try:
        legacy_display = compatibility_decode(counts, compatibility_map)
    except ValueError:
        legacy_display = None
    return {"decision": decision, "legacy_display": legacy_display}

In [52]:
def run_defense_tests():
    groups = {}

    def check(group, ok):
        g = groups.setdefault(group, {"passed": 0, "failed": 0})
        g["passed" if ok else "failed"] += 1

    cases = service.list_cases()
    all_fps = [format(i, "04b") for i in range(16)]

    # 1. Instance cases: the two false rejects (MIG-DELTA, INCIDENT-7) are fixed.
    for c in cases:
        counts = service.execute(c.case_id)["counts"]
        check("instance_cases_match", secure_decision(counts, c.registry_fingerprint) == "MATCH")

    # 2-3. Every 4-bit fingerprint x every map: MATCH on the true registry, MISMATCH on all 15 others.
    for fp in all_fps:
        for perm in permutations(range(4)):
            counts = service.simulate_adapter(fp, perm)["counts"]
            check("true_registry_match_under_any_map", secure_decision(counts, fp) == "MATCH")
            for other in all_fps:
                if other != fp:
                    check("wrong_registry_mismatch", secure_decision(counts, other) == "MISMATCH")

    # 4. Malformed or ambiguous evidence fails closed.
    for bad in ({}, {"0101": 256, "1010": 256}, {"01a1": 512}, {"0101": 300, "10101": 212},
                {"0101": 200, "0111": 180, "1111": 132}):
        check("fail_closed_bad_counts", secure_decision(bad, "1010") == "MISMATCH")
    check("fail_closed_wrong_length_registry", secure_decision({"0101": 512}, "101") == "MISMATCH")

    # 5. A clear majority under noise still decodes; the width is not hard-coded.
    check("noisy_majority_match", secure_decision({"0101": 400, "0111": 112}, "1010") == "MATCH")
    check("five_bit_fingerprint", secure_decision({"00101": 512}, "10100") == "MATCH")

    # 6. Legacy display still exists but cannot change the decision.
    for c in cases:
        counts = service.execute(c.case_id)["counts"]
        report = secure_report(counts, c.registry_fingerprint, service.get_manifest(c.case_id)["compatibility_map"])
        check("legacy_display_cannot_change_decision", report["decision"] == "MATCH")

    return {
        "implemented": True,
        "passed": sum(g["passed"] for g in groups.values()),
        "failed": sum(g["failed"] for g in groups.values()),
        "fingerprints_not_in_instance": sorted(set(all_fps) - {c.registry_fingerprint for c in cases}),
        "groups": groups,
    }

defense_results = run_defense_tests()
defense_results

{'implemented': True,
 'passed': 6160,
 'failed': 0,
 'fingerprints_not_in_instance': ['0000',
  '0001',
  '0010',
  '0011',
  '0100',
  '0101',
  '0111',
  '1000',
  '1011',
  '1101',
  '1110',
  '1111'],
 'groups': {'instance_cases_match': {'passed': 4, 'failed': 0},
  'true_registry_match_under_any_map': {'passed': 384, 'failed': 0},
  'wrong_registry_mismatch': {'passed': 5760, 'failed': 0},
  'fail_closed_bad_counts': {'passed': 5, 'failed': 0},
  'fail_closed_wrong_length_registry': {'passed': 1, 'failed': 0},
  'noisy_majority_match': {'passed': 1, 'failed': 0},
  'five_bit_fingerprint': {'passed': 1, 'failed': 0},
  'legacy_display_cannot_change_decision': {'passed': 4, 'failed': 0}}}

## Phase E — Exploit objective

Assume an attacker can modify only the compatibility permutation in an imported manifest. Determine when a chosen target report is reachable from a source fingerprint.

Your exploit must:

- return a valid permutation when the target is reachable;
- return `None` when it is impossible;
- verify the result through `service.simulate_adapter`;
- explain the limitation imposed by the number of zeroes and ones.

In [53]:
from math import comb


def craft_compatibility_map(source: str, target: str):
    """Permutation p with source[p[i]] == target[i] (the adapter's rule), or None if unreachable."""
    if len(source) != len(target) or set(source + target) - set("01"):
        return None
    if source.count("1") != target.count("1"):
        return None   # a permutation only reorders bits: the number of 1s cannot change
    free = {b: [i for i, x in enumerate(source) if x == b] for b in "01"}
    return tuple(free[b].pop(0) for b in target)


def attack(source, target):
    m = craft_compatibility_map(source, target)
    reported = service.simulate_adapter(source, m)["reported_fingerprint"] if m is not None else None
    return {"source": source, "target": target, "map": m, "reported": reported, "success": reported == target}


all_fps = [format(i, "04b") for i in range(16)]
results = [attack(a, b) for a in all_fps for b in all_fps]
attack_demo = {
    "examples": [attack("1010", "0110"), attack("1100", "0101"), attack("1000", "1100"), attack("0000", "0001")],
    "false_accept": {**attack("1100", "0011"),
                     "meaning": "the asset really encodes 1100; with this map the service reports 0011, "
                                "so it would MATCH a registry entry of 0011 (wrong asset accepted)"},
    "pairs_tested": len(results),
    "reachable_and_verified": sum(r["success"] for r in results),
    "impossible_returned_none": sum(r["map"] is None for r in results),
    "none_only_when_ones_differ": all((r["map"] is None) == (r["source"].count("1") != r["target"].count("1"))
                                      for r in results),
    "reachable_targets_per_source_by_number_of_ones": {k: comb(4, k) for k in range(5)},
    "limitation": ("A permutation only reorders bits, so the target must have the same length and the same number "
                   "of 1s as the source (an anagram): C(4,k) targets for a source with k ones, 70 of 256 pairs; "
                   "0000 and 1111 cannot be changed at all."),
}
attack_demo

{'examples': [{'source': '1010',
   'target': '0110',
   'map': (1, 0, 2, 3),
   'reported': '0110',
   'success': True},
  {'source': '1100',
   'target': '0101',
   'map': (2, 0, 3, 1),
   'reported': '0101',
   'success': True},
  {'source': '1000',
   'target': '1100',
   'map': None,
   'reported': None,
   'success': False},
  {'source': '0000',
   'target': '0001',
   'map': None,
   'reported': None,
   'success': False}],
 'false_accept': {'source': '1100',
  'target': '0011',
  'map': (2, 3, 0, 1),
  'reported': '0011',
  'success': True,
  'meaning': 'the asset really encodes 1100; with this map the service reports 0011, so it would MATCH a registry entry of 0011 (wrong asset accepted)'},
 'pairs_tested': 256,
 'reachable_and_verified': 70,
 'impossible_returned_none': 186,
 'none_only_when_ones_differ': True,
 'reachable_targets_per_source_by_number_of_ones': {0: 1,
  1: 4,
  2: 6,
  3: 4,
  4: 1},
 'limitation': 'A permutation only reorders bits, so the target must have th

## Research extension

Choose at least one:

- Exhaustively test all 4-bit fingerprints and all 24 permutations; quantify how many permutations alter each fingerprint.
- Design a manifest-validation policy that distinguishes security semantics from presentation semantics.
- Add an authenticated binding for security-relevant metadata and explain what it does **not** protect.
- Generalize the decoder and exploit to `n` bits, then discuss scalability.
- Construct a metamorphic test suite that detects order-dependent application behavior without knowing the secret fingerprint in advance.

In [54]:
all_fps = [format(i, "04b") for i in range(16)]
maps = list(permutations(range(4)))

# Extension 1: exhaustive test, how many of the 24 maps alter each fingerprint
altering = {fp: sum(service.simulate_adapter(fp, p)["reported_fingerprint"] != fp for p in maps) for fp in all_fps}
by_ones = {}
for fp, n in altering.items():
    by_ones.setdefault(fp.count("1"), set()).add(n)

# Extension 2: metamorphic test, same counts + only the map changed => the security result must not change
metamorphic = {}
for case in service.list_cases():
    counts = service.execute(case.case_id)["counts"]          # opaque result: the fingerprint is never inspected
    legacy_reports = {compatibility_decode(counts, p) for p in maps}
    secure_decisions = {secure_report(counts, case.registry_fingerprint, p)["decision"] for p in maps}
    metamorphic[case.case_id] = {
        "legacy_distinct_reports": len(legacy_reports),
        "legacy_order_dependent": len(legacy_reports) > 1,
        "secure_distinct_decisions": len(secure_decisions),
        "secure_order_dependent": len(secure_decisions) > 1,
    }

extension_results = {
    "exhaustive": {
        "runs": len(all_fps) * len(maps),
        "altering_maps_per_fingerprint": altering,
        "altering_maps_by_number_of_ones": {k: sorted(v) for k, v in sorted(by_ones.items())},
        "total_altering_pairs": sum(altering.values()),
        "formula": "a fingerprint with k ones is kept by k!(4-k)! maps, so 24 - k!(4-k)! maps alter it",
    },
    "metamorphic": {
        "relation": "same counts, only the compatibility_map changes => the security result must not change",
        "per_case": metamorphic,
    },
}
extension_results

{'exhaustive': {'runs': 384,
  'altering_maps_per_fingerprint': {'0000': 0,
   '0001': 18,
   '0010': 18,
   '0011': 20,
   '0100': 18,
   '0101': 20,
   '0110': 20,
   '0111': 18,
   '1000': 18,
   '1001': 20,
   '1010': 20,
   '1011': 18,
   '1100': 20,
   '1101': 18,
   '1110': 18,
   '1111': 0},
  'altering_maps_by_number_of_ones': {0: [0],
   1: [18],
   2: [20],
   3: [18],
   4: [0]},
  'total_altering_pairs': 264,
  'formula': 'a fingerprint with k ones is kept by k!(4-k)! maps, so 24 - k!(4-k)! maps alter it'},
 'metamorphic': {'relation': 'same counts, only the compatibility_map changes => the security result must not change',
  'per_case': {'CAL-ALPHA': {'legacy_distinct_reports': 6,
    'legacy_order_dependent': True,
    'secure_distinct_decisions': 1,
    'secure_order_dependent': False},
   'CAL-BETA': {'legacy_distinct_reports': 6,
    'legacy_order_dependent': True,
    'secure_distinct_decisions': 1,
    'secure_order_dependent': False},
   'MIG-DELTA': {'legacy_disti

## Claim sheet

Complete this table before preparing the presentation.

| Claim | Evidence cell/output | Assumption | Limitation |
|---|---|---|---|
| The fault is / is not quantum noise | | | |
| The circuit does / does not encode the registry value | | | |
| Metadata can / cannot alter the decision | | | |
| The mitigation generalizes | | | |

In [48]:
causal_claim = ("Changing only the compatibility_map, while holding the encoded fingerprint, "
                "the circuit and the simulator seed constant, changes the reported fingerprint "
                "and the MATCH/MISMATCH decision.")

fp = "1010"
maps = list(permutations(range(4)))
runs = [service.simulate_adapter(fp, p) for p in maps]
controlled_experiment = {
    "fingerprint": fp,
    "maps_tested": len(runs),
    "distinct_counts": sorted({str(r["counts"]) for r in runs}),
    "distinct_canonical": sorted({r["canonical_fingerprint"] for r in runs}),
    "distinct_reported": sorted({r["reported_fingerprint"] for r in runs}),
    "maps_giving_match": [p for p, r in zip(maps, runs) if r["reported_fingerprint"] == fp],
    "noise_check_keys": sorted({k for shots in (1, 512, 4096) for seed in (0, 7, 2026)
                                for k in service.execute("MIG-DELTA", shots=shots, seed=seed)["counts"]}),
}
controlled_experiment

{'fingerprint': '1010',
 'maps_tested': 24,
 'distinct_counts': ["{'0101': 512}"],
 'distinct_canonical': ['1010'],
 'distinct_reported': ['0011', '0101', '0110', '1001', '1010', '1100'],
 'maps_giving_match': [(0, 1, 2, 3), (0, 3, 2, 1), (2, 1, 0, 3), (2, 3, 0, 1)],
 'noise_check_keys': ['0101']}

## Export submission

Fill the fields below. The exported JSON is not a substitute for the completed notebook; it gives the jury a machine-readable index of your claims.

In [20]:
submission = {
    "team_id": service.team_id,
    "challenge": "01-trust-no-input",
    "security_claim": "",
    "root_cause_layer": "",
    "minimal_reproducer": reproducer,
    "defense_results": defense_results,
    "attack_demo": attack_demo,
    "claims": claim_sheet,
    "team_member_contributions": [],
}

Path("submission.json").write_text(json.dumps(submission, indent=2), encoding="utf-8")
print("Wrote submission.json")

Wrote submission.json


## Prepare for oral defense

Every member should be ready to answer at least one of these:

- Why is the dominant Qiskit count key not automatically the canonical logical fingerprint?
- Which controlled experiment excludes circuit noise as the root cause?
- Can every chosen target be produced by permutation alone? Why?
- Why is validating that a map is a permutation insufficient?
- Where should legacy presentation conversion occur?
- What happens if the jury gives you a fresh fingerprint and compatibility map?

### Technical references

- IBM Quantum, *Bit ordering in Qiskit*: https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering
- IBM Quantum, `Statevector.probabilities_dict`: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector
- Qiskit BasicSimulator API: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.providers.basic_provider.BasicSimulator